# Kannolo Sparse Multivector Rerank Index Benchmark (LoTTE)

Build and search a two-stage index: first-stage retrieval with HNSW on sparse vectors, followed by multi-vector (MaxSim) reranking on half-precision (f16) token vectors.

The first stage is built (or loaded) on its own. `kannolo.compose` combines it with the multi-vector data, so one saved first stage can be combined with several rescoring datasets. For compressed multi-vector reranking, replace `kannolo.Plain("f16")` with `kannolo.TwoLevelPQ(...)`.

## Import Required Libraries

In [ ]:
import importlib.metadata as md

print("kannolo", md.version("kannolo"))

In [ ]:
import kannolo
import ir_datasets
import ir_measures
import numpy as np
import pandas as pd
import time
from pathlib import Path
import os

## Load Dataset and Relevance Judgments

In [ ]:
# Load qrels from TSV file
qrels_path = ""
qrels_df = pd.read_csv(qrels_path, sep="\t", header=None, names=["query_id", "Q0", "doc_id", "relevance"])
print(f"Loaded {len(qrels_df)} qrels from {qrels_path}")

# Convert to ir_measures QRel format - map query indices to actual query IDs
queries_ids_path = ""
queries_ids = np.load(queries_ids_path, allow_pickle=True)
print(f"Loaded {len(queries_ids)} query IDs from {queries_ids_path}")

# Map query indices in qrels to actual query IDs
qrels_df['query_id'] = qrels_df['query_id'].apply(lambda x: str(queries_ids[x]))
qrels_df['doc_id'] = qrels_df['doc_id'].astype(str)

qrels = [ir_measures.Qrel(str(row['query_id']), str(row['doc_id']), int(row['relevance']))
         for _, row in qrels_df.iterrows()]

# Get number of queries from queries_ids array
n_queries = len(queries_ids)
print(f"Total queries available: {n_queries}")

## Paths to the First Stage and the Data

In [ ]:
# Sparse documents in seismic binary format (.bin), to build the first stage
sparse_documents_path = ""

# Where the first-stage index is saved with Index.save (and loaded from, if it exists)
sparse_index_path = ""

# Path to multivector data folder (containing documents.npy, doclens.npy, queries.npy)
multivec_data_folder = ""

# Path to sparse queries folder (components.npy, values.npy, offsets.npy)
# If sparse queries are in seismic's bin format, use scripts/convert_bin_to_npy_arrays.py to convert them to numpy arrays
sparse_queries_path = ""

# Path to document ID mapping (maps index to actual doc_id)
doc_ids_path = ""

print(f"Sparse first stage: {sparse_index_path or sparse_documents_path}")
print(f"Loading multivec data from {multivec_data_folder}")
print(f"Loading sparse queries from {sparse_queries_path}")
print(f"Loading doc_ids from {doc_ids_path}")

In [ ]:
# Load sparse queries: a (components, values, offsets) batch
sparse_queries = (
    np.load(Path(sparse_queries_path) / "components.npy"),
    np.load(Path(sparse_queries_path) / "values.npy"),
    np.load(Path(sparse_queries_path) / "offsets.npy"),
)
n_sparse_queries = len(sparse_queries[2]) - 1
print(f"Loaded sparse queries: {n_sparse_queries} queries")

# Load multivector queries, shape (n_queries, n_tokens, token_dim): one (n_tokens, token_dim)
# array per query
multivec_queries = np.load(Path(multivec_data_folder) / "queries.npy").astype(np.float32)
print(f"Loaded multivec queries shape: {multivec_queries.shape}")
multivec_queries = list(multivec_queries)

## Build Rerank Index

In [ ]:
start = time.time()

# First stage: a sparse HNSW (f16 values), loaded if saved before, otherwise built and saved
if sparse_index_path and Path(sparse_index_path).exists():
    first_stage = kannolo.load(sparse_index_path)
else:
    first_stage = kannolo.build(sparse_documents_path, encoder=kannolo.Plain("f16"), index=kannolo.HNSW(m=32, ef_construction=200))
    if sparse_index_path:
        first_stage.save(sparse_index_path)
print(f"First stage ready in {time.time() - start:.2f} seconds: {first_stage}")

# Multi-vector documents: f16 token vectors stored as uint16 bit patterns, plus the number of
# tokens of each document
documents = np.load(Path(multivec_data_folder) / "documents.npy").view(np.float16).astype(np.float32)
doclens = np.load(Path(multivec_data_folder) / "doclens.npy")

# Compose: the first stage is shared, the multi-vector stage is built from the arrays
start = time.time()
rerank_index = kannolo.compose(first_stage, kannolo.Rerank(kannolo.Plain("f16"), data=(documents, doclens)))
print(f"Rerank index composed in {time.time() - start:.2f} seconds: {rerank_index}")

# Load document ID mapping (converts result indices to actual doc_ids)
doc_ids_map = np.load(doc_ids_path, allow_pickle=True)
print(f"Loaded doc_ids mapping with {len(doc_ids_map)} documents")

## Search with Multiple Parameter Combinations

In [ ]:
# Parameter configurations to test
configs = [
    {"k_candidates": 25, "ef_search": 50, "alpha": 0.025},
    {"k_candidates": 100, "ef_search": 100, "alpha": 0.025},
    {"k_candidates": 200, "ef_search": 200, "alpha": 0.1}
]

k = 10  # final top-k results
results = []

for config in configs:
    k_candidates = config["k_candidates"]
    ef_search = config["ef_search"]
    alpha = config["alpha"]

    # Search: sparse queries for the first stage, multi-vector queries for the rescoring stage
    start = time.time()
    dists, ids = rerank_index.search(
        sparse_queries,
        k,
        rerank_queries=multivec_queries,
        k_candidates=k_candidates,
        ef_search=ef_search,
        alpha=alpha,
    )
    elapsed = time.time() - start

    # Build results as DataFrame for ir_measures
    # Map query indices and document indices to actual IDs
    result_rows = []
    for q_idx in range(n_sparse_queries):
        actual_query_id = str(queries_ids[q_idx])
        for rank, (doc_index, score) in enumerate(zip(ids[q_idx], dists[q_idx])):
            if doc_index < 0:  # fewer than k results
                break
            result_rows.append({
                'query_id': actual_query_id,
                'doc_id': str(doc_ids_map[int(doc_index)]),
                'rank': rank,
                'score': float(score)
            })

    res_df = pd.DataFrame(result_rows)

    # Ensure all IDs are strings for ir_measures
    res_df['query_id'] = res_df['query_id'].astype(str)
    res_df['doc_id'] = res_df['doc_id'].astype(str)

    # Evaluate with Success@5 metric using DataFrame
    metric = ir_measures.parse_measure("Success@5")
    metric_value = ir_measures.calc_aggregate([metric], qrels_df, res_df)[metric]
    avg_time = (elapsed / n_sparse_queries) * 10**6  # us per query

    results.append({
        "k_candidates": k_candidates,
        "ef_search": ef_search,
        "alpha": alpha,
        "avg_time_us": avg_time,
        "success_at_5": metric_value
    })

    print(f"k_cand={k_candidates}, ef={ef_search}, alpha={alpha}: "
          f"avg_time={avg_time:.3f}us, Success@5={metric_value:.3f}")

print("\n" + "="*60)
for r in results:
    print(f"Config: k_candidates={r['k_candidates']}, ef_search={r['ef_search']}, alpha={r['alpha']}")
    print(f"  Avg Query Time: {r['avg_time_us']:.3f} us")
    print(f"  Success@5: {r['success_at_5']:.3f}")
    print()